# AIT303 Assignment 1 — Aspect Extraction & Topic Modeling

**Author:** [Student Name]

This notebook performs aspect extraction on scraped Best Buy Bluetooth & Wireless Speaker reviews
using three topic modeling approaches:
- **LDA** (gensim LdaMulticore) — unsupervised
- **BERTopic** (all-MiniLM-L6-v2 embeddings) — unsupervised
- **CorEx** (anchored semi-supervised) — semi-supervised

**Pipeline:** Preprocessing → SpaCy Keyphrase Extraction → LDA/BERTopic → Aspect Restructuring → CorEx

## 1. Colab Setup & Data Loading

### ⚡ Colab Instructions
If running on Google Colab:
1. Upload the `data_asg/bestbuy/` folder to your Google Drive (as `data_asg/bestbuy/`)
2. Set `COLAB = True` in the config cell below
3. Run all cells — the notebook will mount your Drive

**First-run note:** The pip install cell downloads ~200MB of packages. Sentence-transformers
model (all-MiniLM-L6-v2, 80MB) downloads on first BERTopic use. Both cache after first run.

In [ ]:
# ============================================
# CONFIGURATION
# ============================================
COLAB = True

if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/data_asg'
    BESTBUY_DIR = f'{DATA_DIR}/bestbuy'
    MODEL_DIR = f'{DATA_DIR}/models'
else:
    DATA_DIR = 'data_asg'
    BESTBUY_DIR = f'{DATA_DIR}/bestbuy'
    MODEL_DIR = 'models'

print(f"Running in {'COLAB' if COLAB else 'LOCAL'} mode")
print(f"Data directory: {DATA_DIR}")
print(f"Best Buy data:  {BESTBUY_DIR}")
print(f"Model directory: {MODEL_DIR}")

import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')

In [ ]:
# Install all required packages (Colab)
# gensim 4.4.0 cannot install on Python 3.14 - this notebook runs on Colab (Python 3.10)
!pip install gensim==4.4.0 spacy==3.8.0 bertopic==0.17.4 corextopic==1.1 pyLDAvis==3.4.1 umap-learn hdbscan curl-cffi
!pip install --upgrade jupyter-client
!python -m spacy download en_core_web_sm

import warnings
warnings.filterwarnings('ignore')

print("Package installation complete")

In [ ]:
# ============================================
# IMPORTS
# ============================================
import os
import re
import json
import random
import warnings
import pickle
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# NLP
import spacy

# LDA
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

# BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# CorEx
from corextopic import corextopic as ct

# LDA visualization
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Scikit-learn utilities
from sklearn.feature_extraction.text import CountVectorizer as SkCountVectorizer

# Web scraping (TLS impersonation for Best Buy hidden API)
from curl_cffi import requests as curl_requests

# Reproducibility
np.random.seed(42)
random.seed(42)

print("All imports loaded successfully")
print(f"pandas {pd.__version__}, numpy {np.__version__}, spaCy {spacy.__version__}")

## 2. Load & Preprocess Reviews

In [ ]:
# Load the consolidated reviews CSV
csv_path = f'{BESTBUY_DIR}/all_reviews.csv'
df = pd.read_csv(csv_path)

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nUnique products: {df['product_name'].nunique()}")
print(f"Total reviews:   {len(df)}")
print(f"\nFirst 3 rows:")
df.head(3)

In [ ]:
# Load product metadata catalog
catalog_path = f'{BESTBUY_DIR}/products.json'
with open(catalog_path, 'r') as f:
    products = json.load(f)
products_df = pd.DataFrame(products)

print(f"Products in catalog: {len(products_df)}")
print(f"Columns: {list(products_df.columns)}")
products_df.head(5)

In [ ]:
# Reviews per product
review_counts = df['product_name'].value_counts()

plt.figure(figsize=(12, 5))
review_counts.head(20).plot(kind='bar')
plt.title('Reviews per Product (Top 20)')
plt.xlabel('Product')
plt.ylabel('Review Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Min reviews per product: {review_counts.min()}")
print(f"Max reviews per product: {review_counts.max()}")
print(f"Mean reviews per product: {review_counts.mean():.1f}")
print(f"Products with >= 80 reviews: {(review_counts >= 80).sum()}")

### 2.1 Text Preprocessing — Clean Review Text

Reusing the `clean_text()` pipeline from Phase 1 (sentiment_analysis_preprocessing.ipynb):
lowercase → remove HTML → remove non-alpha → normalize whitespace (per D-24).
Product metadata is stored separately and NOT prepended to review text (per D-25).

In [7]:
def clean_text(text):
    """Clean raw review text: lowercase, remove HTML tags, remove non-alpha characters, normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
# Apply Phase 1 preprocessing to review text
df['cleaned_review'] = df['review_text'].apply(clean_text)

# Verify cleaning
print("Before vs After Cleaning (sample):")
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"BEFORE: {df['review_text'].iloc[i][:150]}")
    print(f"AFTER:  {df['cleaned_review'].iloc[i][:150]}")

print(f"\nCleaned review length stats:")
print(df['cleaned_review'].str.len().describe())

# Filter out empty reviews after cleaning
empty_count = (df['cleaned_review'].str.len() == 0).sum()
print(f"\nEmpty reviews after cleaning: {empty_count} ({empty_count/len(df)*100:.1f}%)")

In [ ]:
# Tokenize cleaned reviews for gensim LDA input
# df['tokens'] = df['cleaned_review'].str.split()
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
df['tokens'] = df['cleaned_review'].apply(
    lambda x: [w for w in x.split() if w not in ENGLISH_STOP_WORDS and len(w) > 1]
)

# Filter short reviews (minimum 3 tokens for meaningful topic contribution)
df_valid = df[df['tokens'].apply(len) >= 3].copy()

print(f"Reviews with >= 3 tokens: {len(df_valid)} / {len(df)}")
print(f"Removed {len(df) - len(df_valid)} very short reviews")

In [ ]:
# Save preprocessed DataFrame for downstream use
preprocessed_path = f'{BESTBUY_DIR}/preprocessed_reviews.csv'
df_valid.to_csv(preprocessed_path, index=False)
print(f"Preprocessed data saved to: {preprocessed_path}")
print(f"Shape: {df_valid.shape}")

## 3. Keyphrase Extraction with SpaCy

Extracting meaningful keyphrases (noun chunks and named entities) from the cleaned review text
using SpaCy's linguistic annotations. These keyphrases serve as candidate aspect terms
for downstream topic modeling.

### ⏱ First-Run Note
The cell below downloads the `en_core_web_sm` model (~12MB). This is a one-time download
that caches after the first run.

In [ ]:
# ===== SpaCy Keyphrase Extraction =====
# Extract noun chunks and named entities as candidate aspect terms
nlp = spacy.load('en_core_web_sm')

def extract_keyphrases(text):
    """Extract noun chunks and named entities as aspect-relevant keyphrases."""
    doc = nlp(text)
    keyphrases = set()
    
    # Noun chunks (e.g., "battery life", "sound quality")
    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()
        if len(phrase.split()) <= 4:  # max 4 words
            keyphrases.add(phrase)
    
    # Named entities (e.g., "JBL", "Bose")
    for ent in doc.ents:
        if ent.label_ in ['ORG', 'PRODUCT', 'EVENT']:
            keyphrases.add(ent.text.lower().strip())
    
    return list(keyphrases)


# Test on a sample
sample = df['cleaned_review'].iloc[0]
print(f"Sample text: {sample[:200]}...")
print(f"Keyphrases: {extract_keyphrases(sample)}")


In [ ]:
# ===== Apply Keyphrase Extraction =====
print("Extracting keyphrases from all reviews (this may take a few minutes)...")
df['keyphrases'] = df['cleaned_review'].apply(extract_keyphrases)
print(f"Keyphrases extracted for {len(df)} reviews")
print(f"\nSample keyphrases:")
print(df[['cleaned_review', 'keyphrases']].head(3))


## 4. Unsupervised Aspect Extraction — LDA Topic Modeling

Training LDA with LdaMulticore (multi-core parallelization).
Grid search over 6-10 topic counts, selecting the best via C_v topic coherence (D-27).
Targeting 6-10 interpretable topics covering product aspects (D-29).

In [ ]:
# ===== Build gensim Dictionary and Corpus =====
# Prepare tokenized reviews for gensim
tokenized_reviews = df['tokens'].tolist()

# Build dictionary — maps token → integer ID
dictionary = Dictionary(tokenized_reviews)

# Filter extremes: remove words in <5 docs, keep words in <=50% of all docs
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Create bag-of-words corpus: list of (token_id, frequency) vectors
corpus = [dictionary.doc2bow(tokens) for tokens in tokenized_reviews]

print(f"Dictionary size: {len(dictionary)} unique tokens")
print(f"Corpus size: {len(corpus)} documents")

In [ ]:
# ===== LDA Grid Search — 6 to 10 Topics =====
topic_range = range(6, 11)
coherence_scores = {}
lda_models = {}

for k in topic_range:
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        chunksize=2000,
        passes=10,
        iterations=400,
        alpha='auto',
        eta='auto',
        random_state=42
    )
    cm = CoherenceModel(
        model=lda,
        texts=tokenized_reviews,
        dictionary=dictionary,
        coherence='c_v',
        topn=20
    )
    score = cm.get_coherence()
    coherence_scores[k] = score
    lda_models[k] = lda
    print(f"K={k}: C_v coherence = {score:.4f}")

best_k = max(coherence_scores, key=coherence_scores.get)
best_lda = lda_models[best_k]
print(f"\nBest K={best_k} with C_v = {coherence_scores[best_k]:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
ks = sorted(coherence_scores.keys())
scores = [coherence_scores[k] for k in ks]
plt.plot(ks, scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Topics (K)', fontsize=12)
plt.ylabel('C_v Coherence Score', fontsize=12)
plt.title('LDA Topic Coherence by Topic Count', fontsize=14)
plt.xticks(ks)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Best LDA Model — K={best_k}")
for topic_id in range(best_k):
    top_words = best_lda.show_topic(topic_id, topn=15)
    words_str = ", ".join([f"{word} ({prob:.3f})" for word, prob in top_words])
    print(f"\nTopic {topic_id}: {words_str}")

In [ ]:
vis_data = gensimvis.prepare(best_lda, corpus, dictionary)
pyLDAvis.display(vis_data)

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
lda_path = f'{MODEL_DIR}/lda_model'
best_lda.save(lda_path)
dictionary.save(f'{MODEL_DIR}/lda_dictionary.pkl')
print(f"LDA model saved to: {lda_path}")

## 5. Unsupervised Aspect Extraction — BERTopic

Using BERTopic with all-MiniLM-L6-v2 sentence embeddings (D-28).
Pre-calculating embeddings for reproducibility (per Anti-pattern 4).
UMAP random_state=42 prevents stochastic behavior (per Pitfall 5).
Targeting 6-10 interpretable topics (D-29).
Note: First run downloads the 80MB all-MiniLM-L6-v2 model (~5 min on Colab).

In [ ]:
# ===== BERTopic: Pre-calculate Embeddings =====
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(
    df['cleaned_review'].tolist(),
    show_progress_bar=True,
    batch_size=64
)
print(f"Embedding shape: {embeddings.shape}")

In [18]:
# ===== Configure BERTopic Pipeline =====
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

In [ ]:
# ===== Train BERTopic =====
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True
)
topics, probs = topic_model.fit_transform(df['cleaned_review'].tolist(), embeddings)
topic_info = topic_model.get_topic_info()
print(f"Non-outlier topics: {len(topic_info[topic_info['Topic'] != -1])}")
print(topic_info.head(15))

In [ ]:
# ===== Display Topics =====
for topic_id in topic_info['Topic'].tolist():
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    words_str = ", ".join([f"{word} ({score:.3f})" for word, score in words])
    print(f"Topic {topic_id}: {words_str}")

In [ ]:
fig = topic_model.visualize_topics()
print(type(fig))  # Should be <class 'plotly.graph_objs._figure.Figure'>
print(fig.data)    # Should show trace data, not empty

In [ ]:
# Topic barchart
fig1 = topic_model.visualize_barchart(top_n_topics=min(10, len(topic_info)))
# Inter-topic distance map
fig2 = topic_model.visualize_topics()

from IPython.display import HTML, display
display(HTML(fig1.to_html(include_plotlyjs='cdn')))
display(HTML(fig2.to_html(include_plotlyjs='cdn')))

In [ ]:
topic_model.visualize_barchart(top_n_topics=min(10, len(topic_info)))

In [ ]:
# Inter-topic distance map
topic_model.visualize_topics().show()

In [ ]:
# ===== Save BERTopic Model =====
bertopic_path = f'{MODEL_DIR}/bertopic_model'
topic_model.save(bertopic_path, serialization="safetensors", save_ctfidf=True, save_embedding_model="sentence-transformers/all-MiniLM-L6-v2")
print(f"BERTopic model saved to: {bertopic_path}")

## 6. Topic Analysis & Keyword Extraction

Analyzing keywords from both LDA and BERTopic outputs.
Building keyword inventory for automated keyword-to-aspect mapping
using 8 predefined aspect categories (D-30):
Design, Sound Quality, Battery, Price, Build Quality, Features, Connectivity, Comfort/Portability

In [ ]:
# ===== LDA Top Keywords =====
lda_keywords = {}
for topic_id in range(best_k):
    words = best_lda.show_topic(topic_id, topn=10)
    lda_keywords[topic_id] = [w for w, p in words]
print("LDA Topics — Top 10 Keywords:")
for tid, words in lda_keywords.items():
    print(f"  Topic {tid}: {', '.join(words)}")

In [ ]:
# ===== BERTopic Top Keywords =====
bertopic_keywords = {}
for topic_id in topic_info['Topic'].tolist():
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    bertopic_keywords[topic_id] = [w for w, s in words]
print("BERTopic Topics — Top 10 Keywords:")
for tid, words in bertopic_keywords.items():
    count = topic_info.loc[topic_info['Topic'] == tid, 'Count'].values[0]
    print(f"  Topic {tid} ({count} revs): {', '.join(words)}")

In [ ]:
# ===== Model Comparison =====
print("MODEL COMPARISON — LDA vs BERTopic")
print(f"LDA: best K={best_k}, C_v coherence={coherence_scores[best_k]:.4f}")
non_outlier = len(topic_info[topic_info['Topic'] != -1])
oc = topic_info.loc[topic_info['Topic'] == -1, 'Count'].values[0] if -1 in topic_info['Topic'].values else 0
print(f"BERTopic: {non_outlier} topics, {oc} outlier reviews")
print("Next: Map keywords to 8 predefined aspects, use as CorEx anchors.")

In [ ]:
# ===== Save Keyword Inventory =====
keyword_data = {
    'lda': {str(k): v for k, v in lda_keywords.items()},
    'bertopic': {str(k): v for k, v in bertopic_keywords.items()},
    'lda_coherence': {str(k): float(v) for k, v in coherence_scores.items()},
    'best_lda_k': best_k
}
with open(f'{MODEL_DIR}/topic_keywords.json', 'w') as f:
    json.dump(keyword_data, f, indent=2)
print(f"Keywords saved: {MODEL_DIR}/topic_keywords.json")

## 7. Semi-Supervised Aspect Extraction — CorEx

Mapping LDA/BERTopic keywords to 8 predefined aspect categories (D-30):
1. Design         — look, style, appearance, aesthetic, sleek
2. Sound Quality  — sound, audio, quality, clear, loud, bass, treble
3. Battery        — battery, life, charge, lasting, power, recharge
4. Price          — price, value, cost, cheap, expensive, worth, budget
5. Build Quality  — build, quality, solid, durable, well, sturdy, material
6. Features       — features, app, eq, settings, mode, control, voice
7. Connectivity   — bluetooth, range, signal, connection, stable, pairing
8. Comfort/Portability — portable, light, carry, size, compact, travel, weight

Keywords can belong to multiple aspects (D-32). CorEx anchors derived from
top LDA/BERTopic keywords per aspect (D-33). All 8 aspects anchored (D-34).

In [ ]:
# ===== Build Aspect Keyword Mapping =====
aspect_keywords = {
    'Design': ['design', 'look', 'style', 'aesthetic', 'appearance', 'sleek', 'beautiful', 'elegant', 'modern', 'color'],
    'Sound Quality': ['sound', 'audio', 'quality', 'clear', 'loud', 'bass', 'treble', 'crisp', 'rich', 'volume', 'music', 'acoustic'],
    'Battery': ['battery', 'life', 'charge', 'lasting', 'power', 'recharge', 'hours', 'playback', 'drain'],
    'Price': ['price', 'value', 'cost', 'cheap', 'expensive', 'worth', 'budget', 'money', 'dollar', 'deal', 'affordable'],
    'Build Quality': ['build', 'quality', 'solid', 'durable', 'well', 'sturdy', 'material', 'plastic', 'metal', 'built', 'construction'],
    'Features': ['features', 'app', 'eq', 'settings', 'mode', 'control', 'voice', 'alexa', 'google', 'smart', 'multiroom', 'led'],
    'Connectivity': ['bluetooth', 'range', 'signal', 'connection', 'stable', 'pairing', 'wifi', 'aux', 'nfc', 'wireless', 'connect'],
    'Comfort/Portability': ['portable', 'light', 'carry', 'size', 'compact', 'travel', 'weight', 'small', 'handheld', 'backpack', 'outdoor']
}

# Display aspect definitions
for aspect, keywords in aspect_keywords.items():
    print(f"{aspect:25s}: {', '.join(keywords[:6])}... ({len(keywords)} keywords)")

In [ ]:
# ===== Build CorEx Anchors =====
# For each predefined aspect, find matching LDA/BERTopic keywords
# This merges unsupervised model outputs with predefined aspect structure
anchors = []
aspect_names = list(aspect_keywords.keys())

for aspect in aspect_names:
    # Start with predefined keywords for the aspect
    anchor_words = list(aspect_keywords[aspect])
    
    # Add relevant LDA keywords (where word overlaps with aspect theme)
    for tid, words in lda_keywords.items():
        for w in words:
            if w not in anchor_words and len(anchor_words) < 10:
                anchor_words.append(w)
    
    # Add relevant BERTopic keywords
    for tid, words in bertopic_keywords.items():
        for w in words:
            if w not in anchor_words and len(anchor_words) < 10:
                anchor_words.append(w)
    
    anchors.append(anchor_words[:10])  # cap at 10 keywords per anchor

# Display anchors
print("CorEx Anchors per Aspect:")
for i, (aspect, anchor) in enumerate(zip(aspect_names, anchors)):
    print(f"  {i}. {aspect:25s}: {', '.join(anchor)}")

In [ ]:
# ===== Train CorEx with Anchored Aspects =====
# Build binary BoW matrix (CorEx operates on word presence/absence)
corEx_vectorizer = CountVectorizer(binary=True, max_features=5000, stop_words='english')
X = corEx_vectorizer.fit_transform(df_valid['cleaned_review'].tolist())
feature_words = corEx_vectorizer.get_feature_names_out()

print(f"Document-term matrix shape: {X.shape}")
print(f"Vocabulary size: {len(feature_words)}")

# Train CorEx with anchored topics (D-34: all 8 aspects anchored)
topic_model_corex = ct.Corex(
    n_hidden=len(anchors),  # One per predefined aspect (8)
    seed=42,
    max_iter=500,
    verbose=True
)

topic_model_corex.fit(
    X,
    words=feature_words,
    anchors=anchors,
    anchor_strength=3  # Per Pitfall 6: default 3 is good starting point
)

print(f"\nCorEx training complete.")
print(f"Number of topics: {topic_model_corex.n_hidden}")
print(f"TC (total correlation): {topic_model_corex.tc:.2f}")

In [ ]:
# ===== Analyze CorEx Topics =====
# for i in range(topic_model_corex.n_hidden):
#     topic_words = topic_model_corex.get_topics(n_words=10)
#     # get_topics returns list of (topic_id, [(word, score), ...])
#     words_with_scores = topic_words[i]
#     words_str = ", ".join([f"{w} ({s:.3f})" for w, s in words_with_scores])
#     print(f"Topic {i} ({aspect_names[i] if i < len(aspect_names) else 'Unknown'}): {words_str}")
topic_words = topic_model_corex.get_topics(n_words=10)
for i in range(topic_model_corex.n_hidden):
    topic_words = topic_model_corex.get_topics(n_words=10)
    words_with_scores = topic_words[i]
    words_str = ", ".join([f"{w} ({s:.3f})" for w, s, _ in words_with_scores])  # ← only change
    print(f"Topic {i} ({aspect_names[i] if i < len(aspect_names) else 'Unknown'}): {words_str}")

# Display topic label distribution
labels = topic_model_corex.labels.flatten()
# Map numeric labels to aspect names
label_names = [aspect_names[int(l)] if 0 <= l < len(aspect_names) else 'Unassigned' for l in labels]

from collections import Counter
label_counts = Counter(label_names)
print("\nAspect Label Distribution:")
for aspect, count in label_counts.most_common():
    print(f"  {aspect:25s}: {count:5d} reviews ({count/len(labels)*100:.1f}%)")

In [ ]:
# ===== CorEx Topic Relationships =====
# Topic-topic correlations (which aspects co-occur)
print("Topic-Topic Correlation Matrix:")
correlations = topic_model_corex.tc
print(f"Total correlation explained by topics: {correlations:.2f}")

# Per-document topic probabilities
p_y_given_x = topic_model_corex.p_y_given_x
print(f"\nTopic probability matrix shape: {p_y_given_x.shape}")
print(f"Mean topic probabilities:")
for i in range(topic_model_corex.n_hidden):
    mean_prob = p_y_given_x[:, i].mean()
    print(f"  {i}. {aspect_names[i]:25s}: {mean_prob:.3f}")

## 8. Save Models & Export Aspect Labels for Phase 5

Saving all three aspect extraction models to disk (ABSA-07):
- LDA model (gensim .save format)
- BERTopic model (safetensors — ~20MB vs >500MB pickle)
- CorEx model (pickle — native format)

Exporting aspect-labeled review CSV as Phase 5 contract (D-35).
Phase 5 loads this CSV and joins with best sentiment model predictions.

In [ ]:
# ===== Save LDA Model =====
os.makedirs(MODEL_DIR, exist_ok=True)

lda_save_path = f'{MODEL_DIR}/lda_model'
best_lda.save(lda_save_path)
print(f"LDA model saved: {lda_save_path}")

In [ ]:
# ===== Save BERTopic Model =====
bertopic_path = f'{MODEL_DIR}/bertopic_model'
topic_model.save(
    bertopic_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="sentence-transformers/all-MiniLM-L6-v2"
)
print(f"BERTopic model saved: {bertopic_path}")

In [ ]:
# ===== Save CorEx Model =====
corex_path = f'{MODEL_DIR}/corex_model.pkl'
with open(corex_path, 'wb') as f:
    pickle.dump(topic_model_corex, f)
print(f"CorEx model saved: {corex_path}")

In [ ]:
print(f"df_valid rows:            {len(df_valid)}")
print(f"labels length:            {len(topic_model_corex.labels.flatten())}")
print(f"p_y_given_x rows:         {topic_model_corex.p_y_given_x.shape[0]}")

In [ ]:
# ===== Export Aspect-Labeled Reviews for Phase 5 =====
# Get most likely topic label per document (argmax over 8 aspects)
labels = topic_model_corex.p_y_given_x.argmax(axis=1)

# Map numeric labels to aspect names
aspect_label_names = [aspect_names[int(l)] if 0 <= l < len(aspect_names) else 'Unassigned' for l in labels]

# Get confidence scores: max probability per review
aspect_confidences = topic_model_corex.p_y_given_x.max(axis=1)

# Build Phase 5 contract DataFrame (D-35 schema)
output_df = pd.DataFrame({
    'review_id': range(len(df_valid)),
    'product_name': df_valid['product_name'].values,
    'review_text': df_valid['cleaned_review'].values,
    'aspect_label': aspect_label_names,
    'aspect_confidence': aspect_confidences,
    'review_rating': df_valid['rating'].values,
    'review_date': df_valid['date'].values
})

# Save CSV
csv_path = f'{BESTBUY_DIR}/aspect_labeled_reviews.csv'
output_df.to_csv(csv_path, index=False)

print(f"Saved {len(output_df)} labeled reviews to: {csv_path}")
print(f"\nColumns: {list(output_df.columns)}")
print(f"\nFirst 5 rows:")
print(output_df.head())

In [ ]:
# ===== Verify All Models Saved =====
import os
print("=" * 60)
print("SAVED MODEL FILES")
print("=" * 60)

model_files = [
    ('LDA', lda_save_path, os.path.exists(lda_save_path)),
    ('BERTopic', bertopic_path, os.path.exists(bertopic_path)),
    ('CorEx', corex_path, os.path.exists(corex_path)),
    ('Aspect CSV', csv_path, os.path.exists(csv_path)),
]

for name, path, exists in model_files:
    status = "\u2713" if exists else "\u2717 MISSING"
    size = os.path.getsize(path) if exists else 0
    size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
    print(f"  [{status}] {name:25s} {size_str:>10s}")

all_saved = all(exists for _, _, exists in model_files)
print(f"\n  All models saved: {'YES \u2713' if all_saved else 'NO \u2717'}")

In [ ]:
# ===== Phase 4 Summary =====
print("=" * 70)
print("  PHASE 4: WEB SCRAPING & ASPECT EXTRACTION — COMPLETE")
print("=" * 70)
print(f"\n  Products scraped:    {df_valid['product_name'].nunique()}")
print(f"  Total reviews:       {len(df_valid)}")
print(f"\n  Unsupervised Models:")
print(f"    LDA topics:        {best_k}")
print(f"    BERTopic topics:   {len(topic_info[topic_info['Topic'] != -1])}")
print(f"\n  Semi-Supervised Model:")
print(f"    CorEx aspects:     {topic_model_corex.n_hidden}")
print(f"    TC (total corr.):  {topic_model_corex.tc:.2f}")
print(f"\n  Outputs:")
print(f"    Aspect models:     {MODEL_DIR}/ (3 models)")
print(f"    Labeled reviews:   {csv_path}")
print(f"\n  Next Phase:")
print(f"    Phase 5: Load aspect_labeled_reviews.csv, label with best")
print(f"    sentiment model, compute per-aspect scores, rank products.")
print("=" * 70)